# Evaluación U5: trayectoria y resultado

El notebook realiza una evaluación concisa con saludos, consultas de reglas y reseñas. Requiere pesos entrenados y `GOOGLE_API_KEY`. Ejecutar tras entrenamiento. Revisar salidas reales y registrar cualquier caso donde el modelo no siga la trayectoria esperada.

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv
from src.agent import build_agent
from src.model import model_status
load_dotenv()
assert model_status()['ready'], 'Ejecute primero entrenamiento_u4.ipynb'
assert os.getenv('GOOGLE_API_KEY'), 'Configure GOOGLE_API_KEY en .env'
agent=build_agent()

### Casos de prueba

In [2]:
cases=[
 {'id':'C1','question':'Hola, ¿qué puedes hacer?','expected':set(),'expected_terms':['hola','reseñas']},
 {'id':'C2','question':'¿Qué significa una confianza menor al 65% y qué recomienda el proyecto?','expected':{'consultar_reglas_triaje'},'expected_terms':['65%','humana']},
 {'id':'C3','question':'Clasifica esta reseña: The acting was wonderful and I loved every minute.','expected':{'clasificar_resena'},'expected_terms':['positiva','garantía']},
 {'id':'C4','question':'Clasifica esta reseña y dime qué se hace con una reseña negativa: The film was dull, confusing, and a waste of time.','expected':{'clasificar_resena','consultar_reglas_triaje'},'expected_terms':['negativa','humana']},
]
results=[]
for case in cases:
    tid='eval-'+case['id']
    response=agent.invoke({'messages':[{'role':'user','content':case['question']}]},config={'configurable':{'thread_id':tid}})
    calls=[call['name'] for msg in response['messages'] for call in (getattr(msg,'tool_calls',[]) or [])]
    answer=response['messages'][-1].content
    if isinstance(answer,list): answer_text=' '.join(block.get('text','') for block in answer if isinstance(block,dict))
    else: answer_text=str(answer)
    trajectory=set(calls)==case['expected']
    result_ok=all(term in answer_text.lower() for term in case['expected_terms'])
    result={'caso':case['id'],'herramientas_esperadas':sorted(case['expected']),'herramientas_invocadas':calls,'trayectoria_correcta':trajectory,'resultado_correcto':result_ok,'respuesta':answer_text}
    results.append(result); print(result)
print('Resumen trayectoria:',sum(x['trayectoria_correcta'] for x in results),'/',len(results))
print('Resumen resultados:',sum(x['resultado_correcto'] for x in results),'/',len(results))

C:\Users\kevin\Documents\Codex\2026-11-18\en-x20\outputs\reseñas-triaje-neuroagentico\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'caso': 'C1', 'herramientas_esperadas': [], 'herramientas_invocadas': [], 'trayectoria_correcta': True, 'resultado_correcto': True, 'respuesta': '¡Hola! Soy un asistente especializado en el análisis de reseñas de IMDB. \n\nEsto es lo que puedo hacer por ti:\n\n1. **Clasificar reseñas:** Puedo analizar el texto de una reseña de película para decirte si es **positiva** o **negativa**, además de indicarte el nivel de confianza de la red neuronal local. *Ten en cuenta que una alta probabilidad no es una garantía absoluta.*\n2. **Consultar las reglas de triaje:** Si tienes dudas sobre los criterios del proyecto, los límites de confianza o cuándo se requiere una revisión humana, puedo consultar la documentación oficial para darte una respuesta exacta basada en nuestras normas.\n\n¿En qué te gustaría que te ayude hoy? Puedes pegarme una reseña para analizar o hacerme una pregunta sobre las reglas.'}


{'caso': 'C2', 'herramientas_esperadas': ['consultar_reglas_triaje'], 'herramientas_invocadas': ['consultar_reglas_triaje'], 'trayectoria_correcta': True, 'resultado_correcto': True, 'respuesta': 'Según las reglas del proyecto, una confianza menor al 65% significa que la clasificación se considera **incierta** en esta demostración. \n\nAnte este nivel de incertidumbre, el proyecto recomienda lo siguiente:\n* Derivar la reseña a **revisión o lectura humana**.\n* Recordar que la salida del modelo es una probabilidad estimada y **no una garantía** absoluta sobre la intención del autor o la calidad objetiva de la película.\n\n*(Nota: Estas reglas corresponden a decisiones didácticas de este prototipo y no a políticas reales de IMDB u otra organización).*'}


{'caso': 'C3', 'herramientas_esperadas': ['clasificar_resena'], 'herramientas_invocadas': ['clasificar_resena'], 'trayectoria_correcta': True, 'resultado_correcto': True, 'respuesta': 'La reseña ha sido clasificada como **Positiva** con una confianza y probabilidad positiva de **1.000**. \n\nTen en cuenta que una probabilidad alta no es una garantía absoluta de que la clasificación sea perfecta, ya que siempre existe un margen de incertidumbre en los modelos automatizados.'}


{'caso': 'C4', 'herramientas_esperadas': ['clasificar_resena', 'consultar_reglas_triaje'], 'herramientas_invocadas': ['clasificar_resena', 'consultar_reglas_triaje'], 'trayectoria_correcta': True, 'resultado_correcto': True, 'respuesta': 'La reseña **"The film was dull, confusing, and a waste of time."** ha sido clasificada como **Negativa** con una confianza del 100.0% (probabilidad positiva: 0.000).\n\n*Nota importante:* Recuerda que esta probabilidad estimada por el modelo no es una garantía absoluta ni demuestra la intención real del autor o la calidad objetiva de la película.\n\n---\n\n### ¿Qué se hace con una reseña negativa según las reglas del proyecto?\n* **Derivación:** Las reseñas negativas o inciertas se derivan a lectura humana.\n* **Límites de acción:** El agente **no** borra, oculta ni modifica reseñas.'}
Resumen trayectoria: 4 / 4
Resumen resultados: 4 / 4


### Prueba de memoria y reinicio

In [3]:
tid='memoria-demo'
a=agent.invoke({'messages':[{'role':'user','content':'Me llamo EquipoDemo. Recuerda ese nombre.'}]},config={'configurable':{'thread_id':tid}})
b=agent.invoke({'messages':[{'role':'user','content':'¿Qué nombre te indiqué?'}]},config={'configurable':{'thread_id':tid}})
print('Mismo thread_id:',b['messages'][-1].content)
print('Para reiniciar: use un thread_id nuevo. En la app el botón de reinicio genera uno nuevo.')

Mismo thread_id: [{'type': 'text', 'text': 'Me indicaste que te llamas EquipoDemo.', 'extras': {'signature': 'EnsKeQFpFH0T/xprnzJO+qdzpW1Y+HyfA30UaNwvAAJl3kXymm1iSsSR+BOO8XBsEKju9dbOsRhTXrJ/8IqLPmwzFsx9/K9WolPkiw3PpLlzD7Kv5Zc/vUw1mzvfQUHvHhtmMSdlacbZ/g7DZw57StbiGALS6AC3JCK1SVQ='}}]
Para reiniciar: use un thread_id nuevo. En la app el botón de reinicio genera uno nuevo.
